# Titanic Survival Prediction — End-to-End Executable Analysis

**Project:** Titanic Survival Prediction  
**Pipeline:** data-engineer → data-analyst-eda → ml-model-scientist  
**Date:** 2026-04-26  

This notebook reproduces the entire analysis from raw data loading to final model evaluation in a single, clean, executable workflow. Run all cells top-to-bottom to fully reproduce all results.

### Pipeline Overview
1. Load and save raw data
2. Clean data and engineer features
3. Exploratory data analysis with visualisations
4. Train three classifiers (Logistic Regression, Random Forest, Gradient Boosting)
5. Select best model by validation accuracy, evaluate on test set
6. Extract and explain top 5 feature importances

## Part 1 — Setup and Configuration

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/Users/chen/PycharmProjects/claude-demo/titanic-survival-prediction/02_src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from IPython.display import Image, display

# Project paths
BASE = '/Users/chen/PycharmProjects/claude-demo/titanic-survival-prediction'
RAW_PATH        = f'{BASE}/00_data/raw/titanic_raw.csv'
PROCESSED_PATH  = f'{BASE}/00_data/processed/titanic_cleaned.parquet'
FIGURES_DIR     = f'{BASE}/03_reports/figures'
MODELS_DIR      = f'{BASE}/04_models'

print("Environment ready.")

## Part 2 — Data Engineering

Load raw data from seaborn, apply cleaning and feature engineering, save outputs.

In [ ]:
from data_processing import load_raw_data, clean_and_engineer, save_processed

raw_df = load_raw_data(RAW_PATH)
cleaned_df = clean_and_engineer(raw_df)
save_processed(cleaned_df, PROCESSED_PATH)

print(f"\nCleaned dataset shape: {cleaned_df.shape}")
print(f"Columns: {list(cleaned_df.columns)}")
print(f"\nClass balance:\n{cleaned_df['survived'].value_counts(normalize=True).round(3)}")
cleaned_df.head()

## Part 3 — Exploratory Data Analysis

Key survival patterns by gender, class, age, family size, fare, and title.

In [ ]:
from visualization import (
    plot_survival_by_gender, plot_survival_by_class, plot_age_distribution,
    plot_survival_by_family_size, plot_fare_distribution,
    plot_correlation_heatmap, plot_survival_by_title,
)

df = pd.read_parquet(PROCESSED_PATH)

plot_survival_by_gender(df, f'{FIGURES_DIR}/survival_by_gender.png')
plot_survival_by_class(df, f'{FIGURES_DIR}/survival_by_class.png')
plot_age_distribution(df, f'{FIGURES_DIR}/age_distribution.png')
plot_survival_by_family_size(df, f'{FIGURES_DIR}/survival_by_family_size.png')
plot_fare_distribution(df, f'{FIGURES_DIR}/fare_distribution.png')
plot_survival_by_title(df, f'{FIGURES_DIR}/survival_by_title.png')
plot_correlation_heatmap(df, f'{FIGURES_DIR}/correlation_heatmap.png')
print("All EDA figures saved.")

In [ ]:
# Display key EDA charts inline
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gender
rates_g = df.groupby('sex')['survived'].mean()
axes[0].bar(['Male', 'Female'], rates_g.values * 100, color=['#5b9bd5', '#e07b8f'])
axes[0].set_title('Survival Rate by Gender')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_ylim(0, 100)

# Class
rates_c = df.groupby('pclass')['survived'].mean()
axes[1].bar(['1st', '2nd', '3rd'], rates_c.values * 100, color=['#2ca02c', '#ff7f0e', '#d62728'])
axes[1].set_title('Survival Rate by Class')
axes[1].set_ylabel('Survival Rate (%)')
axes[1].set_ylim(0, 100)

# Title
title_map = {1: 'Mr', 2: 'Miss', 3: 'Mrs', 4: 'Master', 5: 'Rare'}
tmp = df.copy(); tmp['TLabel'] = tmp['Title'].map(title_map)
rates_t = tmp.groupby('TLabel')['survived'].mean().sort_values(ascending=False)
axes[2].bar(rates_t.index, rates_t.values * 100, color='#6baed6')
axes[2].set_title('Survival Rate by Title')
axes[2].set_ylabel('Survival Rate (%)')
axes[2].set_ylim(0, 100)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/eda_summary_panel.png', bbox_inches='tight')
plt.show()
print("EDA summary panel saved.")

## Part 4 — Model Training and Evaluation

Train/Validation/Test split: 70% / 15% / 15%  
Models: Logistic Regression, Random Forest, Gradient Boosting  
Selection criterion: Validation accuracy

In [ ]:
from modeling import (
    split_data, scale_features, train_all_models, select_best_model,
    evaluate_model, plot_feature_importance, plot_roc_curve,
    plot_model_comparison, plot_confusion_matrix, save_model, FEATURE_COLS
)

# Split data
X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)
X_train_sc, X_val_sc, X_test_sc, scaler = scale_features(X_train, X_val, X_test)

# Train all models
results, fitted_models = train_all_models(X_train, y_train, X_val, y_val, X_train_sc, X_val_sc)

# Select best
best_name = select_best_model(results)
best_model = fitted_models[best_name]
# Use unscaled for tree-based, scaled for LR
X_test_best = X_test_sc if best_name == 'Logistic Regression' else X_test

In [ ]:
# Validation metrics comparison table
rows = []
for name, res in results.items():
    row = {'Model': name}
    row.update({k: round(v, 4) for k, v in res['val'].items()})
    rows.append(row)
comparison_df = pd.DataFrame(rows).set_index('Model')
print("Validation Metrics:")
display(comparison_df)

plot_model_comparison(results, f'{FIGURES_DIR}/model_comparison.png')

In [ ]:
# Test set evaluation
test_metrics = evaluate_model(best_model, X_test_best, y_test, label='Test')
test_df = pd.DataFrame([{k: round(v, 4) for k, v in test_metrics.items()}], index=[f'{best_name} (Test)'])
print(f"\nTest Set Results — {best_name}:")
display(test_df)

In [ ]:
plot_roc_curve(best_model, X_test_best, y_test,
               f'{FIGURES_DIR}/roc_curve.png', model_name=best_name)
plot_confusion_matrix(best_model, X_test_best, y_test,
                      f'{FIGURES_DIR}/confusion_matrix.png', model_name=best_name)
print("ROC curve and confusion matrix saved.")

## Part 5 — Top 5 Feature Importances

In [ ]:
fi_df = plot_feature_importance(best_model, FEATURE_COLS,
                                f'{FIGURES_DIR}/feature_importance.png',
                                model_name=best_name)
print("Top 5 Feature Importances:")
display(fi_df.head(5))

explanations = {
    'Title':      'Encodes gender + social status jointly (Mrs/Miss vs Mr). Most discriminative feature.',
    'fare':       'Direct proxy for socioeconomic class and cabin deck proximity to lifeboats.',
    'sex':        'Women survived at 74.2% vs men at 18.9% — the starkest statistical divide.',
    'age':        'Captures "children first" protocol; young children had elevated survival odds.',
    'pclass':     '1st class (62.9%) vs 3rd class (24.2%): cabin location determined lifeboat access.',
}
print("\nFeature Explanations:")
for _, row in fi_df.head(5).iterrows():
    feat = row['Feature']
    print(f"  {feat:12s} ({row['Importance']:.4f}): {explanations.get(feat, '')}")

## Part 6 — Save Final Model

In [ ]:
save_model(best_model, scaler, f'{MODELS_DIR}/titanic_predictor_v1')

print("\n=== Pipeline Complete ===")
print(f"Best Model:    {best_name}")
print(f"Val Accuracy:  {results[best_name]['val']['accuracy']:.4f}")
print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test ROC-AUC:  {test_metrics['roc_auc']:.4f}")
print(f"\nAll outputs saved under: {BASE}")